In [ ]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Subset

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aemri75anno_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aemri75anno.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[0])
token_df = dataset_df_to_token_df(df)

In [3]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"../encoder_training/aemri75anno_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from utils.models import MDN
import copy

from utils.models import SequenceAutoencoder

device = "mps" if torch.backends.mps.is_available() else "cpu"

art_model = SequenceAutoencoder(
    feature_shape=(174, 2),
    latent_dim=1024,
).to(device)
art_model.load_state_dict(torch.load("../encoder_training/ckpts/aemri75anno/art_model_ckpt.pth", map_location=device))
art_model.eval()

ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)
ac_model.load_state_dict(torch.load("../encoder_training/ckpts/aemri75anno/ac_model_ckpt.pth", map_location=device))
ac_model.eval()

SequenceAutoencoder(
  (enc_conv): Sequential(
    (0): Conv1d(30, 512, kernel_size=(5,), stride=(1,), padding=(2,))
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv1d(512, 256, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (7): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
  )
  (enc_pool): AdaptiveAvgPool1d(output_size=1)
  (fc_enc): Sequential(
    (0): Linear(in_features=128, out_features=4096, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4096, out_features=2048, bias=True)
    (3): ReLU()
    (4): Linear(in_features=2048, out_features=1024, bias=True)
  )
  (fc_dec): Sequential(
    (0): Linear(in_features=1024, out_features=2048, bias=True)
    (1): ReLU()
    (2): Linear(i

In [5]:
art_vectors = []
ac_vectors = []
phonemes = []
subjects = []

pbar = tqdm.tqdm(token_df.iterrows(), total=len(token_df))

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]

    articulatory_features = get_aemri75anno_articulatory_features(art_f, start, end, scalers=scalers[subject])
    if articulatory_features is None: continue
    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 174, 2)
    articulatory_features = torch.from_numpy(articulatory_features).float().unsqueeze(0).to(device)
    art_out, art_z = art_model(articulatory_features)

    mfccs = get_mfccs(ac_f, start, end)
    if mfccs is None: continue
    mfccs = torch.from_numpy(mfccs.T).unsqueeze(0).to(device)
    ac_out, ac_z = ac_model(mfccs)

    art_vectors.append(art_z.detach().cpu().numpy().squeeze())
    ac_vectors.append(ac_z.detach().cpu().numpy().squeeze())
    phonemes.append(phone)
    subjects.append(subject)

100%|██████████| 37849/37849 [1:32:05<00:00,  6.85it/s]


In [6]:
import numpy as np
art = np.stack(art_vectors, axis=0)   # (N, T_A, D_A)
ac = np.stack(ac_vectors, axis=0)  # (N, T_X, D_X)

# A: (N, T_A, D_A)
# X: (N, T_X, D_X)
N, D = art.shape
N2, D_X = ac.shape
assert N == N2

art_flat = art.reshape(N, D)
ac_flat = ac.reshape(N, D_X)   # shape (N, x_dim)

print("art_flat:", art_flat.shape, "ac_flat:", ac_flat.shape)

art_flat: (37670, 1024) ac_flat: (37670, 1024)


In [7]:
art_raw = art_flat.astype(np.float32)
ac_raw = ac_flat.astype(np.float32)

N = len(ac_raw)
k = 5

indices = np.arange(N)
np.random.shuffle(indices)
folds = np.array_split(indices, k)

In [8]:
art_dim = art_raw.shape[1]
ac_dim = ac_raw.shape[1]

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------
class AXDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

# ----------------------------------------------------
# Training loop for one model
# ----------------------------------------------------
def train_model(train_loader, val_loader, x_dim, y_dim):
    model = MDN(x_dim=x_dim, y_dim=y_dim, n_components=5, hidden_dim=256).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    best_val = float('inf')
    patience = 10
    counter = 0

    for epoch in range(1, 2001):  # just large upper bound
        # ----- train -----
        model.train()
        train_loss = 0
        for Xb, Yb in train_loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logp = model.log_prob(Xb, Yb)
            loss = -logp.mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(Xb)
        train_loss /= len(train_loader.dataset)

        # ----- validation -----
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for Xb, Yb in val_loader:
                Xb, Yb = Xb.to(device), Yb.to(device)
                loss = -model.log_prob(Xb, Yb).mean()
                val_loss += loss.item() * len(Xb)
        val_loss /= len(val_loader.dataset)

        print(f"Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}")

        # Early stopping check
        if val_loss < best_val - 1e-4:
            best_val = val_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
            print("   ✓ New best model")
        else:
            counter += 1
            print(f"   No improvement ({counter} epochs)")

        if counter >= patience:
            print(f"Early stopping. Best val={best_val:.4f}")
            break

    model.load_state_dict(best_state)
    return model

In [10]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]


    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_ac = StandardScaler()
    sc_art = StandardScaler()

    sc_ac.fit(ac_raw[train_inner])
    sc_art.fit(art_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    ac_norm = sc_ac.transform(ac_raw)
    art_norm = sc_art.transform(art_raw)

    # Dataset for p(ART | AC)
    dataset = AXDataset(ac_norm, art_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=ac_raw.shape[1],
        y_dim=art_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            ac_fold,
            art_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    ART coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_art.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_art_given_ac_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_art_given_ac_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=704.6798, val=570.4202
   ✓ New best model
Epoch 2: train=548.0261, val=548.6797
   ✓ New best model
Epoch 3: train=518.9187, val=501.1428
   ✓ New best model
Epoch 4: train=484.2441, val=451.9564
   ✓ New best model
Epoch 5: train=472.2830, val=456.8637
   No improvement (1 epochs)
Epoch 6: train=472.2564, val=448.2297
   ✓ New best model
Epoch 7: train=458.7886, val=447.6187
   ✓ New best model
Epoch 8: train=457.8276, val=455.8072
   No improvement (1 epochs)
Epoch 9: train=455.5011, val=436.0462
   ✓ New best model
Epoch 10: train=452.5423, val=447.1724
   No improvement (1 epochs)
Epoch 11: train=450.3957, val=437.2295
   No improvement (2 epochs)
Epoch 12: train=451.4534, val=442.4760
   No improvement (3 epochs)
Epoch 13: train=451.9570, val=434.8889
   ✓ New best model
Epoch 14: train=450.3450, val=434.9853
   No improvement (1 epochs)
Epoch 15: train=449.1704, val=438.2961
   No improvement (2 epochs)
Epoch 16: train=449.4606, val=469.7546
   No impro

In [11]:
import pickle
with open("entropies/aemri75anno_art_given_ac.pickle", "wb") as f:
    pickle.dump(H_art_given_ac_mdn, f)

In [12]:
logp_full = np.zeros(N)

for fold in range(k):

    print(f"\n==============================")
    print(f"   FOLD {fold}")
    print(f"==============================")

    test_idx = folds[fold]
    train_idx = np.hstack([
        folds[i] for i in range(k) if i != fold
    ])

    # ----------------------------------------
    # 1. Make internal train/validation split
    # ----------------------------------------

    train_idx_shuffled = train_idx.copy()
    np.random.shuffle(train_idx_shuffled)

    split = int(0.9 * len(train_idx_shuffled))

    train_inner = train_idx_shuffled[:split]
    val_inner = train_idx_shuffled[split:]

    # ----------------------------------------
    # 2. Fit scalers ONLY on outer training set
    # ----------------------------------------

    sc_art = StandardScaler()
    sc_ac = StandardScaler()

    sc_art.fit(art_raw[train_inner])
    sc_ac.fit(ac_raw[train_inner])

    # ----------------------------------------
    # 3. Transform all data using these scalers
    # ----------------------------------------

    art_norm = sc_art.transform(art_raw)
    ac_norm = sc_ac.transform(ac_raw)

    # Dataset for p(AC | ART)
    dataset = AXDataset(art_norm, ac_norm)

    train_loader = DataLoader(
        Subset(dataset, train_inner),
        batch_size=64,
        shuffle=True
    )

    val_loader = DataLoader(
        Subset(dataset, val_inner),
        batch_size=64,
        shuffle=False
    )

    # ----------------------------------------
    # 4. Train MDN
    # ----------------------------------------

    model = train_model(
        train_loader,
        val_loader,
        x_dim=art_raw.shape[1],
        y_dim=ac_raw.shape[1],
    )

    # ----------------------------------------
    # 5. Evaluate outer held-out fold
    # ----------------------------------------

    model.eval()

    with torch.no_grad():

        art_fold = torch.from_numpy(
            art_norm[test_idx]
        ).float().to(device)

        ac_fold = torch.from_numpy(
            ac_norm[test_idx]
        ).float().to(device)

        logp_standardized = model.log_prob(
            art_fold,
            ac_fold
        ).cpu().numpy()

    # ----------------------------------------
    # 6. Convert density back to original
    #    AC coordinate system
    # ----------------------------------------

    log_jacobian = np.log(sc_ac.scale_).sum()

    logp_original = logp_standardized - log_jacobian

    logp_full[test_idx] = logp_original

phonemes_arr = np.array(phonemes)

H_ac_given_art_mdn = {}

for ph in np.unique(phonemes_arr):

    idx = np.where(phonemes_arr == ph)[0]

    if len(idx) < 30:
        continue

    H_ac_given_art_mdn[ph] = -logp_full[idx].mean()


   FOLD 0
Epoch 1: train=149.0433, val=20.9957
   ✓ New best model
Epoch 2: train=39.2777, val=-20.7271
   ✓ New best model
Epoch 3: train=-8.1268, val=5.5741
   No improvement (1 epochs)
Epoch 4: train=-4.7930, val=-63.2084
   ✓ New best model
Epoch 5: train=-29.8858, val=-25.1799
   No improvement (1 epochs)
Epoch 6: train=9.4997, val=-15.6260
   No improvement (2 epochs)
Epoch 7: train=-9.4684, val=-2.3173
   No improvement (3 epochs)
Epoch 8: train=-25.2644, val=-67.7134
   ✓ New best model
Epoch 9: train=-43.5671, val=-77.4991
   ✓ New best model
Epoch 10: train=-49.5090, val=-65.9532
   No improvement (1 epochs)
Epoch 11: train=-54.2527, val=-63.2946
   No improvement (2 epochs)
Epoch 12: train=-50.7269, val=-56.5934
   No improvement (3 epochs)
Epoch 13: train=-58.2590, val=-65.9106
   No improvement (4 epochs)
Epoch 14: train=-59.0483, val=-57.5626
   No improvement (5 epochs)
Epoch 15: train=-58.9198, val=-79.7164
   ✓ New best model
Epoch 16: train=-62.2184, val=-75.4010
   

In [13]:
with open("entropies/aemri75anno_ac_given_art.pickle", "wb") as f:
    pickle.dump(H_ac_given_art_mdn, f)